In [3]:
import requests
import pandas as pd

rockets = requests.get("https://api.spacexdata.com/v4/rockets").json()
rocket_map = {r["id"]: r["name"] for r in rockets}

launchpads = requests.get("https://api.spacexdata.com/v4/launchpads").json()
launchpad_map = {l["id"]: l["name"] for l in launchpads}

launches = requests.get("https://api.spacexdata.com/v4/launches").json()

payload_cache = {}

records = []

for launch in launches:
    rocket_id = launch.get("rocket")
    launchpad_id = launch.get("launchpad")

    if rocket_id == "5e9d0d95eda69973a809d1ec":

        payload_ids = launch.get("payloads", [])

        masses = []
        orbits = []

        for pid in payload_ids:

            if pid in payload_cache:
                payload = payload_cache[pid]
            else:
                payload = requests.get(
                    f"https://api.spacexdata.com/v4/payloads/{pid}"
                ).json()
                payload_cache[pid] = payload

            masses.append(payload.get("mass_kg"))
            orbits.append(payload.get("orbit"))

        records.append({
            "FlightNumber": launch.get("flight_number"),
            "Date": launch.get("date_utc"),
            "BoosterVersion": rocket_map.get(rocket_id),
            "PayloadMass": sum([m for m in masses if m]),
            "Orbit": orbits[0] if orbits else None,
            "LaunchSite": launchpad_map.get(launchpad_id),
            "Outcome": launch.get("success")
        })

df = pd.DataFrame(records)

df.to_csv("../data/dataset_part_1.csv", index=False)

df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome
0,6,2010-06-04T18:45:00.000Z,Falcon 9,0.0,LEO,CCSFS SLC 40,True
1,7,2010-12-08T15:43:00.000Z,Falcon 9,0.0,LEO,CCSFS SLC 40,True
2,8,2012-05-22T07:44:00.000Z,Falcon 9,525.0,LEO,CCSFS SLC 40,True
3,9,2012-10-08T00:35:00.000Z,Falcon 9,800.0,ISS,CCSFS SLC 40,True
4,10,2013-03-01T19:10:00.000Z,Falcon 9,677.0,ISS,CCSFS SLC 40,True
